# Auditoría rápida de difusión (`auditoria_diffusion`)

Diagnóstico visual del proceso de difusión **sin reentrenar**. Perfil rápido: 2 sujetos test, ~5–15 min GPU.

| Sección | Contenido |
|---------|-----------|
| **1** | Offset coseno `s` — schedule forward + sensibilidad |
| **2** | One-step `predict_x0` vs DDIM vs partial-DDIM |
| **3** | Calibración latente: raw / scale-only / center-only / full |
| **4** | ε-error vs t, amplificación, checklist de mejoras |

**Baseline:** `exp_nm_s2_frozen_h05_roi3ch`. Usa calibración `(z-μ)/σ` antes de difundir y desnormaliza antes del decoder.


In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID";
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm.auto import tqdm

REPO = Path.cwd()
if not (REPO / "config.py").is_file():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
# El codigo del repo asume cwd = raiz (config usa rutas relativas: ROI mask, checkpoints, data).
os.chdir(REPO)

# Jupyter precarga el modulo estandar `code`, que tapa el paquete code/ del repo.
# Lo expulsamos para que `import code.*` resuelva el paquete local (roi_mask_utils, etc.).
sys.modules.pop("code", None)

import config
from code.diffusion_audit_utils import (
    REPO_ROOT,
    amplification_factor,
    apply_calibration,
    build_test_loader,
    calibration_mean,
    calibration_modes,
    calibration_sigma,
    cosine_schedule,
    ddim_sample,
    effective_t,
    first_t_snr_below,
    forward_noisy,
    image_metrics,
    invert_calibration,
    latent_metrics,
    load_audit_models,
    mri_condition,
    one_step_denoise,
    partial_ddim_sample,
    pick_test_subjects,
    schedule_summary_table,
    snr_at_t,
    snr_curve,
    subject_id_from_name,
    tensor_mid_slice,
)
from main import Diffusion
from utils import seed_torch

# --- Knobs ---
AUDIT_EXP = "exp_nm_s2_frozen_h05_roi3ch"
MAX_SUBJECTS = 2
FIXED_T_VALUES = [200, 500, 700, 900]
DDIM_STEPS_LIST = [20, 50, 100]
PARTIAL_DTIM_START = 700
PARTIAL_DTIM_STEPS_LIST = [10, 20, 50]
COSINE_S_VALUES = [0.0, 0.008, 0.02, 0.05]
CALIBRATION_T_HIGH = [700, 900]
SEED = 42
EXPORT_DIR = REPO_ROOT / "result" / "auditoria_diffusion"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device(config.device)
seed_torch(SEED)
sns.set_theme(style="whitegrid")
print(f"Device: {device} | Export: {EXPORT_DIR}")


In [ ]:
models = load_audit_models(AUDIT_EXP, device)
aae = models["aae"]
unet = models["unet"]
mri_encoder = models["mri_encoder"]
diffusion = models["diffusion"]
meta = models["meta"]

LATENT_MEAN = meta["latent_mean"]
LATENT_STD = meta["latent_std"]
USE_LABEL = meta["use_train_label"]
TRAIN_COSINE_S = meta["cosine_s_train"]

subjects_df = pick_test_subjects(n_per_class=1)
loader = build_test_loader(max_subjects=MAX_SUBJECTS)

print("Experiment:", AUDIT_EXP)
print(f"Latent calibration: mean={LATENT_MEAN:.4f}, std={LATENT_STD:.4f}")
print(f"UNet in_ch={meta['in_channels']} roi={meta['use_roi_channel']} label={USE_LABEL}")
subjects_df


## 1. Influencia del offset coseno `s`

**1A** — Curvas del schedule (sin UNet). **1B** — Forward visual en latente real. **1C** — One-step con schedule de forward alternativo (exploratorio; el UNet fue entrenado con `s=0.008`).


In [ ]:
# 1A — Schedule curves
T = config.noiseSteps
t_axis = np.arange(T)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
sigma_raw = LATENT_STD

for s in COSINE_S_VALUES:
    beta, alpha_hat = cosine_schedule(T, s)
    label = f"s={s}"
    axes[0].plot(t_axis, beta, label=label, linewidth=1.5)
    axes[1].plot(t_axis, alpha_hat, label=label, linewidth=1.5)
    axes[2].plot(t_axis, np.sqrt(1 - alpha_hat), label=label, linewidth=1.5)
    axes[3].plot(t_axis, snr_curve(alpha_hat, 1.0), label=f"{label} (σ=1)", linewidth=1.5)

for t_mark in FIXED_T_VALUES + [999]:
    if t_mark < T:
        for ax in axes.flat:
            ax.axvline(t_mark, color="0.8", linestyle=":", linewidth=0.7)

axes[0].set_title(r"$\beta_t$"); axes[0].set_xlabel("t")
axes[1].set_title(r"$\bar{\alpha}_t$"); axes[1].set_xlabel("t")
axes[2].set_title(r"$\sqrt{1-\bar{\alpha}_t}$"); axes[2].set_xlabel("t")
axes[3].set_title("SNR (σ=1, calibrado)"); axes[3].set_xlabel("t")
axes[3].set_yscale("log")
axes[0].legend(fontsize=7, loc="upper left")
plt.suptitle("Comparación de schedules coseno por offset s", y=1.02)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "s1_schedule_curves.png", dpi=150, bbox_inches="tight")
plt.show()

sched_tbl = schedule_summary_table(COSINE_S_VALUES, FIXED_T_VALUES + [999], T, 1.0, sigma_raw)
sched_tbl.round(4).to_csv(EXPORT_DIR / "s1_schedule_table.csv", index=False)
sched_tbl.pivot_table(index="t", columns="cosine_s", values="snr_cal").round(2)


In [ ]:
# 1B — Forward visual (mismo epsilon por sujeto, distintos s y t)
batch = next(iter(loader))
MRI_np, Abeta_np, name, label = batch[0], batch[1], batch[2], batch[3]
sid = subject_id_from_name(name)
MRI_t = torch.as_tensor(np.expand_dims(MRI_np, 1), dtype=torch.float32, device=device)
spect_gt = torch.as_tensor(np.expand_dims(Abeta_np, 1), dtype=torch.float32, device=device)
label_t = label.to(device)

with torch.no_grad():
    z_raw = aae.encoder(spect_gt)
    z_cal = diffusion.normalize_latent(z_raw)
    eps_fixed = torch.randn_like(z_cal)

t_show = [200, 500, 700, 900]
s_show = COSINE_S_VALUES
fig, axes = plt.subplots(len(s_show), len(t_show), figsize=(3 * len(t_show), 2.8 * len(s_show)))
if len(s_show) == 1:
    axes = np.expand_dims(axes, 0)
if len(t_show) == 1:
    axes = np.expand_dims(axes, 1)

for i, s in enumerate(s_show):
    diff_s = Diffusion(cosine_s=s)
    for j, t_val in enumerate(t_show):
        x_t = forward_noisy(diff_s, z_cal, t_val, eps_fixed)
        z_vis = diffusion.denormalize_latent(x_t)
        sl = tensor_mid_slice(z_vis)
        ax = axes[i, j]
        im = ax.imshow(sl, cmap="viridis")
        ax.set_title(f"s={s}, t={t_val}")
        ax.axis("off")
        if j == 0:
            ax.set_ylabel(f"s={s}", fontsize=9)

plt.suptitle(f"Forward latente — sujeto {sid} (slice axial)", y=1.01)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "s1_forward_latent_grid.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# 1C — One-step MSE vs (s_forward, t) — UNet entrenado con s=0.008
MRI_np, Abeta_np, name, label = next(iter(loader))
MRI_t = torch.as_tensor(np.expand_dims(MRI_np, 1), dtype=torch.float32, device=device)
spect_gt = torch.as_tensor(np.expand_dims(Abeta_np, 1), dtype=torch.float32, device=device)
label_t = label.to(device)
sid = subject_id_from_name(name)
cond, precomputed = mri_condition(mri_encoder, MRI_t)
rows_s = []

with torch.no_grad():
    z_raw = aae.encoder(spect_gt)
    z_cal = diffusion.normalize_latent(z_raw)
    eps_fixed = torch.randn_like(z_cal)

    for s_fwd in COSINE_S_VALUES:
        diff_fwd = Diffusion(cosine_s=s_fwd)
        for t_val in FIXED_T_VALUES:
            z_hat_cal, _, _ = one_step_denoise(
                diffusion, unet, z_cal, t_val, cond, label_t, precomputed,
                USE_LABEL, eps=eps_fixed, forward_diff=diff_fwd,
            )
            z_hat_raw = diffusion.denormalize_latent(z_hat_cal)
            m = latent_metrics(z_hat_raw, z_raw)
            rows_s.append({"cosine_s_forward": s_fwd, "t": t_val, **m})

s_heatmap_df = pd.DataFrame(rows_s)
pivot = s_heatmap_df.pivot(index="t", columns="cosine_s_forward", values="latent_mse")
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax)
ax.set_title(f"One-step latent MSE vs s_forward (UNet s={TRAIN_COSINE_S}) — {sid}")
plt.tight_layout()
plt.savefig(EXPORT_DIR / "s1_onestep_mse_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
s_heatmap_df.to_csv(EXPORT_DIR / "s1_onestep_by_s.csv", index=False)


## 2. One-step vs DDIM vs partial-DDIM

Calibración **full** antes de difundir; métricas en espacio latente **crudo** vs `z_gt`.


In [ ]:
diag_rows = []
subject_cache = {}

for batch_idx, (MRI_np, Abeta_np, name, label) in enumerate(tqdm(loader, desc="sampling audit")):
    sid = subject_id_from_name(name)
    MRI_t = torch.as_tensor(np.expand_dims(MRI_np, 1), dtype=torch.float32, device=device)
    spect_gt = torch.as_tensor(np.expand_dims(Abeta_np, 1), dtype=torch.float32, device=device)
    label_t = label.to(device)

    with torch.no_grad():
        z_raw = aae.encoder(spect_gt)
        z_cal = diffusion.normalize_latent(z_raw)
        cond, precomputed = mri_condition(mri_encoder, MRI_t)
        eps_subj = torch.randn_like(z_cal)
        subject_cache[sid] = (spect_gt, z_raw, z_cal, cond, label_t, precomputed, eps_subj)

        diag_rows.append({"ID": sid, "method": "oracle", "t_or_steps": None, **latent_metrics(z_raw, z_raw), **image_metrics(aae, spect_gt, z_raw)})

        for t_val in FIXED_T_VALUES:
            z_hat_cal, eps_true, eps_pred = one_step_denoise(
                diffusion, unet, z_cal, t_val, cond, label_t, precomputed,
                USE_LABEL, eps=eps_subj,
            )
            z_hat_raw = diffusion.denormalize_latent(z_hat_cal)
            eps_mse = float(torch.mean((eps_pred - eps_true) ** 2).item())
            m = {**latent_metrics(z_hat_raw, z_raw), **image_metrics(aae, spect_gt, z_hat_raw), "eps_mse": eps_mse}
            diag_rows.append({"ID": sid, "method": "fixed_t", "t_or_steps": t_val, **m})

        for steps in DDIM_STEPS_LIST:
            z_gen = ddim_sample(
                diffusion, unet, cond, label_t, precomputed, USE_LABEL,
                sampling_steps=steps, seed=SEED + batch_idx,
            )
            m = {**latent_metrics(z_gen, z_raw), **image_metrics(aae, spect_gt, z_gen)}
            diag_rows.append({"ID": sid, "method": "ddim", "t_or_steps": steps, **m})

        for steps in PARTIAL_DTIM_STEPS_LIST:
            z_part = partial_ddim_sample(
                diffusion, unet, z_cal, PARTIAL_DTIM_START, cond, label_t,
                precomputed, USE_LABEL, sampling_steps=steps, eps=eps_subj,
                seed=SEED + batch_idx + 100,
            )
            m = {**latent_metrics(z_part, z_raw), **image_metrics(aae, spect_gt, z_part)}
            diag_rows.append({"ID": sid, "method": "partial_ddim", "t_or_steps": steps, **m})

diag_df = pd.DataFrame(diag_rows)
diag_df.to_csv(EXPORT_DIR / "s2_methods_comparison.csv", index=False)
diag_df.head(10)


In [ ]:
# 2 — Plots
summary = (
    diag_df.groupby(["method", "t_or_steps"], dropna=False)
    .agg(latent_mse_mean=("latent_mse", "mean"), latent_corr_mean=("latent_corr", "mean"),
         PSNR_mean=("PSNR", "mean"), SSIM_mean=("SSIM", "mean"), n=("ID", "count"))
    .reset_index()
)
summary.to_csv(EXPORT_DIR / "s2_summary.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ft = diag_df[diag_df["method"] == "fixed_t"]
if not ft.empty:
    g = ft.groupby("t_or_steps")["latent_mse"].mean()
    axes[0].plot(g.index, g.values, "o-", label="fixed_t")
dd = diag_df[diag_df["method"] == "ddim"]
if not dd.empty:
    g = dd.groupby("t_or_steps")["latent_mse"].mean()
    axes[0].plot(g.index, g.values, "s-", label="ddim")
pd_df = diag_df[diag_df["method"] == "partial_ddim"]
if not pd_df.empty:
    g = pd_df.groupby("t_or_steps")["latent_mse"].mean()
    axes[0].plot(g.index, g.values, "^-", label=f"partial from t={PARTIAL_DTIM_START}")
axes[0].set_xlabel("t or steps"); axes[0].set_ylabel("latent MSE (mean)"); axes[0].legend(); axes[0].set_title("MSE por método")

# Bar compare at t=700 vs DDIM-100
bar_rows = []
for label, filt in [
    ("one-step@700", diag_df[(diag_df.method=="fixed_t") & (diag_df.t_or_steps==700)]),
    ("DDIM-100", diag_df[(diag_df.method=="ddim") & (diag_df.t_or_steps==100)]),
    (f"partial@{PARTIAL_DTIM_START}-50", diag_df[(diag_df.method=="partial_ddim") & (diag_df.t_or_steps==50)]),
]:
    if not filt.empty:
        bar_rows.append({"method": label, "latent_mse": filt["latent_mse"].mean()})
if bar_rows:
    bar_df = pd.DataFrame(bar_rows)
    axes[1].bar(bar_df["method"], bar_df["latent_mse"], color=["steelblue", "coral", "seagreen"][:len(bar_df)])
    axes[1].set_ylabel("latent MSE"); axes[1].tick_params(axis="x", rotation=15)
    axes[1].set_title("Comparación t alto")
plt.tight_layout()
plt.savefig(EXPORT_DIR / "s2_method_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Visual panel for first subject
sid0 = diag_df["ID"].iloc[1] if len(diag_df) > 1 else diag_df["ID"].iloc[0]
spect_gt, z_raw, z_cal, cond, label_t, precomputed, eps_subj = subject_cache[sid0]
with torch.no_grad():
    z_hat_cal, _, _ = one_step_denoise(diffusion, unet, z_cal, 700, cond, label_t, precomputed, USE_LABEL, eps=eps_subj)
    z_onestep = diffusion.denormalize_latent(z_hat_cal)
    z_ddim = ddim_sample(diffusion, unet, cond, label_t, precomputed, USE_LABEL, sampling_steps=100, seed=SEED)
    dec_gt = aae.decoder(z_raw)[0, 0].cpu().numpy()
    dec_os = aae.decoder(z_onestep)[0, 0].cpu().numpy()
    dec_dd = aae.decoder(z_ddim)[0, 0].cpu().numpy()
z_mid = dec_gt.shape[2] // 2
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, img, title in zip(axes, [dec_gt, dec_os, dec_dd, np.abs(dec_gt - dec_dd)],
                          ["GT decode", "one-step@700", "DDIM-100", "|error|"]):
    ax.imshow(img[:, :, z_mid], cmap="viridis")
    ax.set_title(title); ax.axis("off")
plt.suptitle(f"SPECT axial — {sid0}")
plt.tight_layout()
plt.savefig(EXPORT_DIR / "s2_spect_panel.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Calibración: raw / scale-only / center-only / full

Compara el forward (SNR) y one-step @ t altos bajo cada modo de calibración.


In [ ]:
# 3A — SNR curves per calibration mode
_, alpha_hat_train = cosine_schedule(config.noiseSteps, TRAIN_COSINE_S)
t_axis = np.arange(len(alpha_hat_train))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for mode in calibration_modes():
    sig = calibration_sigma(mode, LATENT_STD)
    axes[0].plot(t_axis, snr_curve(alpha_hat_train, sig), label=f"{mode} (σ_eff={sig:.2f})")
    t_below = first_t_snr_below(alpha_hat_train, sig)
    if t_below:
        axes[0].axvline(t_below, linestyle="--", alpha=0.4)
axes[0].set_yscale("log"); axes[0].set_xlabel("t"); axes[0].set_ylabel("SNR")
axes[0].set_title("SNR por modo de calibración"); axes[0].legend(fontsize=8)

# histogram (primer sujeto del loader)
MRI_np, Abeta_np, _, _ = next(iter(loader))
spect_demo = torch.as_tensor(np.expand_dims(Abeta_np, 1), dtype=torch.float32, device=device)
with torch.no_grad():
    z_raw_demo = aae.encoder(spect_demo)
for mode in calibration_modes():
    z_p = apply_calibration(z_raw_demo, mode, LATENT_MEAN, LATENT_STD).cpu().numpy().ravel()
    axes[1].hist(z_p, bins=60, alpha=0.45, density=True, label=f"{mode} μ={z_p.mean():.2f} σ={z_p.std():.2f}")
axes[1].set_title("Distribución z' (1 sujeto)"); axes[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "s3_calibration_forward.png", dpi=150, bbox_inches="tight")
plt.show()

# 3B — One-step @ high t per calibration mode
cal_rows = []
for batch_idx, (MRI_np, Abeta_np, name, label) in enumerate(loader):
    sid = subject_id_from_name(name)
    MRI_t = torch.as_tensor(np.expand_dims(MRI_np, 1), dtype=torch.float32, device=device)
    spect_gt = torch.as_tensor(np.expand_dims(Abeta_np, 1), dtype=torch.float32, device=device)
    label_t = label.to(device)
    cond, precomputed = mri_condition(mri_encoder, MRI_t)
    with torch.no_grad():
        z_raw = aae.encoder(spect_gt)
        eps_c = torch.randn_like(z_raw)
        for mode in calibration_modes():
            z_in = apply_calibration(z_raw, mode, LATENT_MEAN, LATENT_STD)
            eps_in = torch.randn_like(z_in)
            for t_val in CALIBRATION_T_HIGH:
                z_hat_cal, _, _ = one_step_denoise(
                    diffusion, unet, z_in, t_val, cond, label_t, precomputed,
                    USE_LABEL, eps=eps_in,
                )
                z_hat_raw = invert_calibration(z_hat_cal, mode, LATENT_MEAN, LATENT_STD)
                m = latent_metrics(z_hat_raw, z_raw)
                cal_rows.append({"ID": sid, "calibration": mode, "t": t_val, **m})

cal_df = pd.DataFrame(cal_rows)
cal_df.to_csv(EXPORT_DIR / "s3_calibration_onestep.csv", index=False)
pivot_cal = cal_df.groupby(["calibration", "t"])["latent_mse"].mean().unstack()
fig, ax = plt.subplots(figsize=(8, 4))
pivot_cal.plot(kind="bar", ax=ax)
ax.set_ylabel("latent MSE mean"); ax.set_title("One-step @ t alto por modo calibración")
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(EXPORT_DIR / "s3_calibration_mse_bars.png", dpi=150, bbox_inches="tight")
plt.show()
pivot_cal.round(4)


## 4. Análisis adicionales (t altos)

ε-error vs t, factor de amplificación, partial-DDIM sensitivity.


In [ ]:
# 4 — epsilon MSE vs t + amplification overlay
_, ah = cosine_schedule(config.noiseSteps, TRAIN_COSINE_S)
amp = [amplification_factor(ah, t) for t in FIXED_T_VALUES]
eps_by_t = diag_df[(diag_df.method == "fixed_t") & (diag_df.t_or_steps.isin(FIXED_T_VALUES))]
eps_summary = eps_by_t.groupby("t_or_steps").agg(eps_mse=("eps_mse", "mean"), latent_mse=("latent_mse", "mean")).reset_index()

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(eps_summary["t_or_steps"], eps_summary["eps_mse"], "o-", color="tab:blue", label="MSE(ε_pred, ε_true)")
ax1.set_xlabel("t"); ax1.set_ylabel("ε MSE", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(FIXED_T_VALUES, amp, "s--", color="tab:red", label="√(1-ᾱ)/√ᾱ")
ax2.set_ylabel("Amplificación z0", color="tab:red")
ax1.set_title("Error ε vs t y factor de amplificación")
fig.legend(loc="upper left", bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.savefig(EXPORT_DIR / "s4_eps_vs_amplification.png", dpi=150, bbox_inches="tight")
plt.show()

# Partial DDIM steps sensitivity
part = diag_df[diag_df.method == "partial_ddim"]
if not part.empty:
    fig, ax = plt.subplots(figsize=(6, 4))
    g = part.groupby("t_or_steps")["latent_mse"].mean()
    ax.plot(g.index, g.values, "o-")
    ax.set_xlabel(f"DDIM steps from t={PARTIAL_DTIM_START}"); ax.set_ylabel("latent MSE")
    ax.set_title("Partial-DDIM: pasos vs calidad")
    plt.tight_layout()
    plt.savefig(EXPORT_DIR / "s4_partial_ddim_steps.png", dpi=150, bbox_inches="tight")
    plt.show()


## Checklist de decisiones

| Pregunta | Cómo leer este notebook |
|----------|-------------------------|
| ¿Cambiar `s`? | Si SNR@t=700 cambia mucho entre `s` y el heatmap 1C empeora fuera de 0.008 → **reentrenar E2** antes de adoptar otro `s`. |
| ¿Calibración full? | Sección 3: `full` debe dar el menor MSE one-step@700 si el UNet se entrenó con calibración ON. |
| ¿One-step vs DDIM? | Sección 2: si `fixed_t@700` >> `DDIM-100`, el cuello de botella es la cadena de denoising / ε en t altos. |
| ¿Partial-DDIM? | Si partial@700 con pocos pasos acerca a DDIM completo, conviene más pasos desde t alto que solo one-step. |

**Mejoras a explorar (requieren cambio de código / retrain):**
- v-prediction o x0-prediction en lugar de ε
- Min-SNR-γ weighting en la pérdida
- Sobremuestreo `HIGH_T_SAMPLE_PROB` + `LATENT_LOSS` en t>700
- Reentrenar con `cosine_s` alternativo tras validar en sección 1

Export CSV/PNG en `result/auditoria_diffusion/`.
